# HealthCast V2 — Robust 7-Day COVID Forecasting

V2 is a clean experimental pipeline built from the raw state-wise COVID dataset.

V1 is frozen. V2 addresses the problems discovered in V1 without weakening the evaluation:

- diagnose the train/test level shift first
- compare persistence and 7-day seasonal-naive baselines
- keep leakage-safe temporal features
- test multiple model families
- select models using validation only
- keep the final chronological test untouched until final evaluation
- require both final-test improvement and rolling robustness before saving a V2 production model
- never overwrite V1 artifacts

In [1]:
# 1 — Setup

from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
STATE_CODES = ["TN", "KA", "MH", "DL", "KL"]
HORIZON = 7

# Put the uploaded raw file at:
# F:/HealthCast-main/data/state_data.csv
HERE = Path.cwd()
candidates = [
    HERE / "data" / "state_data.csv",
    HERE.parent / "data" / "state_data.csv",
    HERE / "state_data.csv",
    HERE.parent / "state_data.csv",
]

DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find state_data.csv. Put the uploaded file at "
        "F:/HealthCast-main/data/state_data.csv. Checked: "
        + str([str(p) for p in candidates])
    )

PROJECT_ROOT = HERE.parent if HERE.name == "notebooks" else HERE
ARTIFACT_DIR_V2 = PROJECT_ROOT / "artifacts_v2"
ARTIFACT_DIR_V2.mkdir(parents=True, exist_ok=True)

print("DATA_PATH:", DATA_PATH)
print("ARTIFACT_DIR_V2:", ARTIFACT_DIR_V2)

DATA_PATH: f:\HealthCast-main\state_data.csv
ARTIFACT_DIR_V2: f:\HealthCast-main\artifacts_v2


In [2]:
# 2 — Load raw state-wise COVID data

raw = pd.read_csv(DATA_PATH)

print("Shape:", raw.shape)
print("Columns:", raw.columns.tolist())
print("\nStatus counts:")
display(raw["Status"].value_counts(dropna=False))
display(raw.head())

Shape: (1524, 42)
Columns: ['Date', 'Date_YMD', 'Status', 'TT', 'AN', 'AP', 'AR', 'AS', 'BR', 'CH', 'CT', 'DN', 'DD', 'DL', 'GA', 'GJ', 'HR', 'HP', 'JK', 'JH', 'KA', 'KL', 'LA', 'LD', 'MP', 'MH', 'MN', 'ML', 'MZ', 'NL', 'OR', 'PY', 'PB', 'RJ', 'SK', 'TN', 'TG', 'TR', 'UP', 'UT', 'WB', 'UN']

Status counts:


Status
Confirmed    508
Recovered    508
Deceased     508
Name: count, dtype: int64

,Date,Date_YMD,Status,TT,AN,AP,AR,AS,BR,CH,...,PB,RJ,SK,TN,TG,TR,UP,UT,WB,UN
0,14-Mar-20,2020-03-14,Confirmed,81,0,1,0,0,0,0,...,1,3,0,1,1,0,12,0,0,0
1,14-Mar-20,2020-03-14,Recovered,9,0,0,0,0,0,0,...,0,1,0,0,0,0,4,0,0,0
2,14-Mar-20,2020-03-14,Deceased,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,15-Mar-20,2020-03-15,Confirmed,27,0,0,0,0,0,0,...,0,1,0,0,2,0,1,0,0,0
4,15-Mar-20,2020-03-15,Recovered,4,0,0,0,0,0,0,...,0,2,0,0,1,0,0,0,0,0


In [3]:
# 3 — Build daily confirmed-case series

date_col = "Date_YMD" if "Date_YMD" in raw.columns else "Date"

df = raw.copy()
df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df = df[df["Status"].astype(str).str.strip().str.lower().eq("confirmed")].copy()

for st in STATE_CODES:
    df[st] = pd.to_numeric(df[st], errors="coerce").fillna(0.0)

df = df[[date_col] + STATE_CODES].dropna(subset=[date_col])

state_daily = (
    df.groupby(date_col)[STATE_CODES]
      .sum()
      .sort_index()
)

# Make the date index continuous.
full_dates = pd.date_range(state_daily.index.min(), state_daily.index.max(), freq="D")
state_daily = state_daily.reindex(full_dates).fillna(0.0)
state_daily.index.name = "Date"

print("Daily table:", state_daily.shape)
display(state_daily.head())
display(state_daily.tail())

Daily table: (508, 5)


,TN,KA,MH,DL,KL
Date,,,,,
2020-03-14,1,6,14,7,19
2020-03-15,0,0,18,0,5
2020-03-16,0,1,6,0,3
2020-03-17,0,2,3,1,0
2020-03-18,1,5,3,2,0


,TN,KA,MH,DL,KL
Date,,,,,
2021-07-30,1947,1890,6600,63,20772
2021-07-31,1986,1987,6959,58,20624
2021-08-01,1990,1875,6479,85,20728
2021-08-02,1957,1285,4869,51,13984
2021-08-03,1908,1674,6005,50,23676


## 4. Feature engineering

Features use only information available before the forecast origin.

V2 retains V1's core lag/rolling features and adds level/trend features to help a model represent changing regimes.

Targets remain original daily case counts; V2 does not cap the target.

In [4]:
# 4 — Leakage-safe temporal features

FEATURES = [
    "lag_1", "lag_2", "lag_3", "lag_7", "lag_14",
    "rolling_avg_7", "rolling_avg_14", "rolling_std_7",
    "growth_rate", "level_ratio_7", "level_ratio_14",
    "trend_slope_7", "trend_slope_14"
]

def slope(values):
    values = np.asarray(values, dtype=float)
    if len(values) < 2 or not np.all(np.isfinite(values)):
        return np.nan
    x = np.arange(len(values), dtype=float)
    return np.polyfit(x, values, 1)[0]

def make_features(series):
    s = pd.Series(series, dtype=float).copy()
    s.index = pd.to_datetime(s.index)
    s = s.sort_index()

    f = pd.DataFrame(index=s.index)
    f["lag_1"] = s.shift(1)
    f["lag_2"] = s.shift(2)
    f["lag_3"] = s.shift(3)
    f["lag_7"] = s.shift(7)
    f["lag_14"] = s.shift(14)

    f["rolling_avg_7"] = s.shift(1).rolling(7).mean()
    f["rolling_avg_14"] = s.shift(1).rolling(14).mean()
    f["rolling_std_7"] = s.shift(1).rolling(7).std()

    prev2 = s.shift(2).replace(0, np.nan)
    f["growth_rate"] = ((s.shift(1) - s.shift(2)) / prev2)
    f["growth_rate"] = f["growth_rate"].replace([np.inf, -np.inf], np.nan).clip(-5, 5)

    f["level_ratio_7"] = s.shift(1) / f["rolling_avg_7"].replace(0, np.nan)
    f["level_ratio_14"] = s.shift(1) / f["rolling_avg_14"].replace(0, np.nan)

    f["trend_slope_7"] = s.shift(1).rolling(7).apply(slope, raw=True)
    f["trend_slope_14"] = s.shift(1).rolling(14).apply(slope, raw=True)

    return f

def make_direct_supervised(series, horizon=7):
    s = pd.Series(series, dtype=float)
    X = make_features(s)
    Y = pd.DataFrame(
        {f"h_{h}": s.shift(-h) for h in range(1, horizon + 1)},
        index=s.index,
    )
    joined = X.join(Y).dropna()
    return joined[FEATURES], joined[[f"h_{h}" for h in range(1, horizon + 1)]]

In [5]:
# 5 — Chronological split and evaluation helpers

def chronological_split(X, Y, test_fraction=0.20):
    n = len(X)
    test_n = max(HORIZON, int(np.ceil(n * test_fraction)))
    split = n - test_n
    return X.iloc[:split], X.iloc[split:], Y.iloc[:split], Y.iloc[split:], split

def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

def persistence_forecast(series, origin_pos, horizon=7):
    return np.repeat(float(series.iloc[origin_pos]), horizon)

def seasonal_naive_forecast(series, origin_pos, horizon=7, season=7):
    values = series.iloc[:origin_pos + 1].to_numpy(dtype=float)
    out = []
    for h in range(1, horizon + 1):
        idx = len(values) - season + h - 1
        idx = min(max(idx, 0), len(values) - 1)
        out.append(values[idx])
    return np.asarray(out, dtype=float)

## 6. Diagnose the V1 failure

This section is intentionally before model training.

It measures the level of the training and final-test periods so we can see whether V1's huge errors are associated with a regime/scale change.

In [6]:
# 6 — Train/test regime diagnostics

diag_rows = []

for st in STATE_CODES:
    series = state_daily[st].astype(float)
    X, Y = make_direct_supervised(series, HORIZON)
    Xtr, Xte, Ytr, Yte, split = chronological_split(X, Y)

    train_dates = Xtr.index
    test_dates = Xte.index

    train_values = series.loc[:train_dates[-1]]
    test_values = series.loc[test_dates[0]:]

    diag_rows.append({
        "state": st,
        "train_end": str(train_dates[-1].date()),
        "test_start": str(test_dates[0].date()),
        "train_mean": float(train_values.mean()),
        "test_mean": float(test_values.mean()),
        "last_train_value": float(train_values.iloc[-1]),
        "test_median": float(test_values.median()),
        "test_to_train_mean_ratio": float(
            test_values.mean() / max(train_values.mean(), 1e-9)
        ),
    })

diagnostics_df = pd.DataFrame(diag_rows).set_index("state")
display(diagnostics_df)

,train_end,test_start,train_mean,test_mean,last_train_value,test_median,test_to_train_mean_ratio
state,,,,,,,
TN,2021-04-20,2021-04-21,2514.585608,14781.657143,10986.0,12772.0,5.878367
KA,2021-04-20,2021-04-21,2974.302730,16298.228571,21794.0,8249.0,5.479680
MH,2021-04-20,2021-04-21,9827.193548,22482.942857,62097.0,10442.0,2.287829
DL,2021-04-20,2021-04-21,2247.000000,5056.285714,28395.0,259.0,2.250238
KL,2021-04-21,2021-04-22,3205.594059,20712.394231,22414.0,17473.5,6.461328


## 7. Baselines

V2 uses two simple baselines:

- **Persistence:** repeat the latest observed value.
- **Seasonal naive:** use the corresponding value from the previous 7-day cycle.

The strongest baseline is the benchmark the V2 model must beat.

In [7]:
# 7 — Baseline final-test evaluation

baseline_rows = []

for st in STATE_CODES:
    series = state_daily[st].astype(float)
    X, Y = make_direct_supervised(series, HORIZON)
    Xtr, Xte, Ytr, Yte, split = chronological_split(X, Y)

    true = Yte.to_numpy(float)

    p = np.vstack([
        persistence_forecast(series, series.index.get_loc(d), HORIZON)
        for d in Xte.index
    ])

    sn = np.vstack([
        seasonal_naive_forecast(series, series.index.get_loc(d), HORIZON)
        for d in Xte.index
    ])

    baseline_rows.append({
        "state": st,
        "persistence_mae": mae(true, p),
        "persistence_rmse": rmse(true, p),
        "seasonal_naive_mae": mae(true, sn),
        "seasonal_naive_rmse": rmse(true, sn),
    })

baseline_df = pd.DataFrame(baseline_rows).set_index("state")
display(baseline_df)

,persistence_mae,persistence_rmse,seasonal_naive_mae,seasonal_naive_rmse
state,,,,
TN,2289.447522,3170.548388,4048.709913,4942.091438
KA,3224.900875,4891.465394,5031.026239,6946.867637
MH,3516.160350,5589.463085,4418.889213,6450.179974
DL,1225.690962,2504.283365,1922.478134,3498.627768
KL,4010.033873,5399.762806,3606.867452,4811.857463


## 8. Candidate models

The candidates intentionally use different modeling assumptions:

1. Random Forest — V1-compatible nonlinear tree ensemble.
2. HistGradientBoosting — boosted nonlinear model.
3. Ridge on `log1p(y)` — linear model with target transformation, allowing a different response to changing scale.

All candidates predict all seven future days directly.

In [8]:
# 8 — Model factories

MODEL_NAMES = [
    "random_forest",
    "hist_gradient_boosting",
    "ridge_log1p",
]

def build_model(name):
    if name == "random_forest":
        return RandomForestRegressor(
            n_estimators=400,
            max_depth=12,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    if name == "hist_gradient_boosting":
        return MultiOutputRegressor(
            HistGradientBoostingRegressor(
                max_iter=300,
                learning_rate=0.05,
                max_leaf_nodes=31,
                l2_regularization=1.0,
                random_state=RANDOM_STATE,
            )
        )

    if name == "ridge_log1p":
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", MultiOutputRegressor(Ridge(alpha=10.0))),
        ])

    raise ValueError(name)

def fit_predict(name, X_train, Y_train, X_pred):
    model = build_model(name)

    if name == "ridge_log1p":
        model.fit(X_train, np.log1p(Y_train.clip(lower=0)))
        pred = np.expm1(model.predict(X_pred))
    else:
        model.fit(X_train, Y_train)
        pred = model.predict(X_pred)

    return model, np.maximum(np.asarray(pred, dtype=float), 0.0)

## 9. Rolling-origin validation

Four chronological validation windows are used.

**Important:** the final test is not used here.

A candidate is considered validation-eligible only if it beats the stronger baseline in at least 3 of 4 windows. Among eligible candidates, the lowest average validation MAE is selected.

In [9]:
# 9 — Rolling-origin validation

def rolling_validate(series, n_windows=4, horizon=7, min_train=60):
    X, Y = make_direct_supervised(series, horizon)
    n = len(X)

    latest_origin = n - horizon
    earliest_origin = max(min_train, latest_origin - (n_windows - 1) * horizon)
    origins = np.linspace(
        earliest_origin,
        latest_origin,
        n_windows,
        dtype=int,
    )

    rows = []

    for model_name in MODEL_NAMES:
        for origin in origins:
            X_train = X.iloc[:origin]
            Y_train = Y.iloc[:origin]
            X_val = X.iloc[origin:origin + horizon]
            Y_val = Y.iloc[origin:origin + horizon]

            if len(X_val) < horizon:
                continue

            _, pred = fit_predict(model_name, X_train, Y_train, X_val)
            true = Y_val.to_numpy(float)

            p = np.vstack([
                persistence_forecast(
                    series, series.index.get_loc(d), horizon
                )
                for d in X_val.index
            ])

            sn = np.vstack([
                seasonal_naive_forecast(
                    series, series.index.get_loc(d), horizon
                )
                for d in X_val.index
            ])

            rows.append({
                "model": model_name,
                "origin": str(X_val.index[0].date()),
                "mae": mae(true, pred),
                "persistence_mae": mae(true, p),
                "seasonal_naive_mae": mae(true, sn),
            })

    return pd.DataFrame(rows)

rolling_tables = {
    st: rolling_validate(state_daily[st].astype(float))
    for st in STATE_CODES
}

rolling_all = pd.concat(
    rolling_tables,
    names=["state", "row"]
).reset_index(drop=True)

display(rolling_all)

,model,origin,mae,persistence_mae,seasonal_naive_mae
0,random_forest,2021-06-30,237.725003,659.673469,1182.530612
1,random_forest,2021-07-07,405.204034,443.204082,901.959184
2,random_forest,2021-07-14,376.959467,253.938776,543.306122
3,random_forest,2021-07-21,176.229481,92.224490,179.551020
4,hist_gradient_boosting,2021-06-30,407.650627,659.673469,1182.530612
5,hist_gradient_boosting,2021-07-07,402.426768,443.204082,901.959184
6,hist_gradient_boosting,2021-07-14,231.422206,253.938776,543.306122
7,hist_gradient_boosting,2021-07-21,116.048592,92.224490,179.551020
8,ridge_log1p,2021-06-30,973.520300,659.673469,1182.530612
9,ridge_log1p,2021-07-07,634.667254,443.204082,901.959184


In [10]:
# 10 — Select a V2 candidate per state using validation only

selection_rows = []

for st in STATE_CODES:
    d = rolling_tables[st].copy()

    summary = d.groupby("model").agg(
        validation_mae=("mae", "mean"),
    )

    d["strongest_baseline_mae"] = d[
        ["persistence_mae", "seasonal_naive_mae"]
    ].min(axis=1)

    wins = (
        d.assign(win=d["mae"] < d["strongest_baseline_mae"])
         .groupby("model")["win"]
         .sum()
    )

    tested = d.groupby("model").size()

    summary["windows_won"] = wins
    summary["windows_tested"] = tested
    summary["rolling_pass"] = (
        (summary["windows_won"] >= 3)
        & (summary["windows_tested"] == 4)
    )

    eligible = summary[summary["rolling_pass"]]

    if len(eligible):
        selected = eligible["validation_mae"].idxmin()
    else:
        selected = None

    selection_rows.append({
        "state": st,
        "selected_model": selected,
        "validation_mae": (
            float(summary.loc[selected, "validation_mae"])
            if selected is not None else np.nan
        ),
        "windows_won": (
            int(summary.loc[selected, "windows_won"])
            if selected is not None else 0
        ),
        "windows_tested": (
            int(summary.loc[selected, "windows_tested"])
            if selected is not None else 0
        ),
        "rolling_pass": bool(selected is not None),
    })

selection_df = pd.DataFrame(selection_rows).set_index("state")
display(selection_df)

,selected_model,validation_mae,windows_won,windows_tested,rolling_pass
state,,,,,
TN,hist_gradient_boosting,289.387049,3,4,True
KA,random_forest,279.689910,3,4,True
MH,None,NaN,0,0,False
DL,None,NaN,0,0,False
KL,None,NaN,0,0,False


## 10. Final untouched test

The selected V2 model is now evaluated once on the chronological final test.

No model choice is made from this result.

The model must beat the stronger of persistence and seasonal naive.

In [11]:
# 11 — Final V2 test

final_rows = []
final_predictions = {}

for st in STATE_CODES:
    series = state_daily[st].astype(float)
    X, Y = make_direct_supervised(series, HORIZON)
    Xtr, Xte, Ytr, Yte, split = chronological_split(X, Y)

    true = Yte.to_numpy(float)

    p = np.vstack([
        persistence_forecast(series, series.index.get_loc(d), HORIZON)
        for d in Xte.index
    ])

    sn = np.vstack([
        seasonal_naive_forecast(series, series.index.get_loc(d), HORIZON)
        for d in Xte.index
    ])

    p_mae = mae(true, p)
    sn_mae = mae(true, sn)
    strongest = min(p_mae, sn_mae)

    selected = selection_df.loc[st, "selected_model"]
    rolling_pass = bool(selection_df.loc[st, "rolling_pass"])

    if selected is None:
        final_rows.append({
            "state": st,
            "selected_model": None,
            "model_mae": np.nan,
            "model_rmse": np.nan,
            "persistence_mae": p_mae,
            "seasonal_naive_mae": sn_mae,
            "strongest_baseline_mae": strongest,
            "beats_strongest_baseline": False,
            "rolling_pass": rolling_pass,
            "production_gate": False,
        })
        continue

    model, pred = fit_predict(selected, Xtr, Ytr, Xte)
    model_mae = mae(true, pred)
    model_rmse = rmse(true, pred)
    beats = model_mae < strongest
    gate = bool(beats and rolling_pass)

    final_rows.append({
        "state": st,
        "selected_model": selected,
        "model_mae": model_mae,
        "model_rmse": model_rmse,
        "persistence_mae": p_mae,
        "seasonal_naive_mae": sn_mae,
        "strongest_baseline_mae": strongest,
        "beats_strongest_baseline": beats,
        "rolling_pass": rolling_pass,
        "production_gate": gate,
    })

    final_predictions[st] = {
        "model": model,
        "predictions": pred,
        "actual": true,
        "test_dates": list(Xte.index),
    }

final_test_df = pd.DataFrame(final_rows).set_index("state")
display(final_test_df)

,selected_model,model_mae,model_rmse,persistence_mae,seasonal_naive_mae,strongest_baseline_mae,beats_strongest_baseline,rolling_pass,production_gate
state,,,,,,,,,
TN,hist_gradient_boosting,9445.693522,13552.894248,2289.447522,4048.709913,2289.447522,False,True,False
KA,random_forest,7259.864727,11446.057419,3224.900875,5031.026239,3224.900875,False,True,False
MH,None,NaN,NaN,3516.160350,4418.889213,3516.160350,False,False,False
DL,None,NaN,NaN,1225.690962,1922.478134,1225.690962,False,False,False
KL,None,NaN,NaN,4010.033873,3606.867452,3606.867452,False,False,False


In [12]:
# 12 — Horizon-level diagnostics

horizon_rows = []

for st, info in final_predictions.items():
    true = info["actual"]
    pred = info["predictions"]
    series = state_daily[st].astype(float)

    dates = info["test_dates"]

    p = np.vstack([
        persistence_forecast(series, series.index.get_loc(d), HORIZON)
        for d in dates
    ])

    sn = np.vstack([
        seasonal_naive_forecast(series, series.index.get_loc(d), HORIZON)
        for d in dates
    ])

    for h in range(HORIZON):
        horizon_rows.append({
            "state": st,
            "horizon": h + 1,
            "model_mae": mae(true[:, h], pred[:, h]),
            "persistence_mae": mae(true[:, h], p[:, h]),
            "seasonal_naive_mae": mae(true[:, h], sn[:, h]),
        })

horizon_df = pd.DataFrame(horizon_rows)
display(horizon_df)

,state,horizon,model_mae,persistence_mae,seasonal_naive_mae
0,TN,1,9438.275896,613.561224,4199.428571
1,TN,2,9612.212172,1201.632653,4151.959184
2,TN,3,9713.143054,1768.438776,4098.795918
3,TN,4,9466.005504,2312.275510,4044.397959
4,TN,5,9267.126321,2847.408163,3995.887755
5,TN,6,9265.052313,3381.561224,3949.244898
6,TN,7,9358.039394,3901.255102,3901.255102
7,KA,1,9077.223857,2011.846939,5399.163265
8,KA,2,8478.927871,2317.367347,5290.408163
9,KA,3,7935.240339,2658.142857,5168.795918


In [13]:
# 13 — Prediction bias diagnostics

bias_rows = []

for st, info in final_predictions.items():
    true = info["actual"]
    pred = info["predictions"]

    bias_rows.append({
        "state": st,
        "model_bias": float(np.mean(pred - true)),
        "actual_mean": float(np.mean(true)),
        "prediction_mean": float(np.mean(pred)),
        "actual_median": float(np.median(true)),
        "prediction_median": float(np.median(pred)),
    })

bias_df = pd.DataFrame(bias_rows).set_index("state")
display(bias_df)

,model_bias,actual_mean,prediction_mean,actual_median,prediction_median
state,,,,,
TN,-8910.314109,15223.778426,6313.464317,14016.0,6950.725445
KA,-6931.424404,16290.769679,9359.345276,8029.5,7542.150668


## 13. Frozen V1 comparison

These are the broad chronological V1 test MAEs already measured. They are **not** used to tune V2.

V2 is successful only if it genuinely improves against the same test regime and also passes the V2 reliability gate.

In [14]:
# 14 — V1 frozen benchmark

V1_BROAD_TEST_MAE = {
    "TN": 10497.213733,
    "KA": 11284.369522,
    "MH": 6741.327720,
    "DL": 2494.220057,
    "KL": 12471.667340,
}

comparison = final_test_df[
    [
        "selected_model",
        "model_mae",
        "persistence_mae",
        "seasonal_naive_mae",
        "production_gate",
    ]
].copy()

comparison["v1_mae"] = pd.Series(V1_BROAD_TEST_MAE)
comparison["v2_vs_v1_pct"] = (
    (comparison["v1_mae"] - comparison["model_mae"])
    / comparison["v1_mae"]
    * 100
)

display(comparison)

,selected_model,model_mae,persistence_mae,seasonal_naive_mae,production_gate,v1_mae,v2_vs_v1_pct
state,,,,,,,
TN,hist_gradient_boosting,9445.693522,2289.447522,4048.709913,False,10497.213733,10.017136
KA,random_forest,7259.864727,3224.900875,5031.026239,False,11284.369522,35.664419
MH,None,NaN,3516.160350,4418.889213,False,6741.327720,NaN
DL,None,NaN,1225.690962,1922.478134,False,2494.220057,NaN
KL,None,NaN,4010.033873,3606.867452,False,12471.667340,NaN


## 14. Final production gate

A state is V2-production eligible only if:

1. its selected model passes 3/4 rolling validation windows, **and**
2. its selected model beats the strongest simple baseline on the untouched final test.

If no state passes, V2 does **not** replace V1. That is an acceptable and scientifically honest outcome.

In [15]:
# 15 — Final gate

production_summary = final_test_df[
    [
        "selected_model",
        "model_mae",
        "persistence_mae",
        "seasonal_naive_mae",
        "beats_strongest_baseline",
        "rolling_pass",
        "production_gate",
    ]
].copy()

display(production_summary)

passed_states = production_summary.index[
    production_summary["production_gate"].fillna(False)
].tolist()

print("V2 production-passed states:", passed_states)

if not passed_states:
    print("NO STATE PASSED THE V2 PRODUCTION GATE.")
    print("Keep V1 artifacts unchanged and treat V2 as an experiment.")
else:
    print("V2 candidates:", passed_states)

,selected_model,model_mae,persistence_mae,seasonal_naive_mae,beats_strongest_baseline,rolling_pass,production_gate
state,,,,,,,
TN,hist_gradient_boosting,9445.693522,2289.447522,4048.709913,False,True,False
KA,random_forest,7259.864727,3224.900875,5031.026239,False,True,False
MH,None,NaN,3516.160350,4418.889213,False,False,False
DL,None,NaN,1225.690962,1922.478134,False,False,False
KL,None,NaN,4010.033873,3606.867452,False,False,False


V2 production-passed states: []
NO STATE PASSED THE V2 PRODUCTION GATE.
Keep V1 artifacts unchanged and treat V2 as an experiment.


## 15. Save V2 artifacts safely

Only states that pass the final gate receive a model file.

`artifacts_v2/` is separate from V1 `artifacts/`.

In [16]:
# 16 — Save V2 artifacts

metadata = {
    "version": "HealthCast_V2",
    "forecast_horizon": HORIZON,
    "feature_columns": FEATURES,
    "data_source": str(DATA_PATH),
    "gate_rule": {
        "final_test_beats_strongest_baseline": True,
        "rolling_windows_required": 3,
        "rolling_windows_tested": 4,
    },
    "states": {},
}

for st in STATE_CODES:
    row = production_summary.loc[st]

    state_meta = {}
    for k, v in row.to_dict().items():
        state_meta[k] = v.item() if isinstance(v, np.generic) else v

    state_meta["rolling_windows_won"] = int(selection_df.loc[st, "windows_won"])
    state_meta["rolling_windows_tested"] = int(selection_df.loc[st, "windows_tested"])
    metadata["states"][st] = state_meta

    if bool(row["production_gate"]) and st in final_predictions:
        joblib.dump(
            final_predictions[st]["model"],
            ARTIFACT_DIR_V2 / f"state_model_{st}.pkl",
        )

with open(ARTIFACT_DIR_V2 / "model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

state_daily.to_csv(ARTIFACT_DIR_V2 / "state_data.csv")

print("Saved metadata:", ARTIFACT_DIR_V2 / "model_metadata.json")
print("Saved source copy:", ARTIFACT_DIR_V2 / "state_data.csv")
print("Production-passed states:", passed_states)

Saved metadata: f:\HealthCast-main\artifacts_v2\model_metadata.json
Saved source copy: f:\HealthCast-main\artifacts_v2\state_data.csv
Production-passed states: []


# Final interpretation

Do not judge V2 by whether it produces forecasts for every state.

Judge it by whether it earns the right to produce a forecast.

- If V2 passes: use only the passed V2 states and update the app.
- If V2 improves but fails the gate: keep it as an experiment.
- If V2 fails similarly to V1: investigate the data-generating process/time-series structure instead of endlessly tuning models.

**Never manually edit metadata to turn a failed model into a passing model.**